# HQViT Data-Efficiency Study — Clean Research Pipeline

**Purpose:** Rebuild the experiment workflow in a reproducible notebook while preserving the original notebook and its results.

**Important:** This notebook is intentionally a clean foundation. It does not import or overwrite old checkpoints automatically. It standardizes the CIFAR-10 split, verifies data integrity, and provides a classical ViT sanity test before any quantum experiments.

## Research safeguards
- CIFAR-10 official test set is held out until final evaluation.
- One fixed train/validation split is reused across models.
- Validation samples are excluded from training subsets.
- The classical model must pass a small-subset learning sanity check before HQViT comparisons.
- Do not interpret short smoke tests as final benchmark results.


## 1. Configuration and reproducibility

In [ ]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
DATA_ROOT = Path("data")
RESULTS_DIR = Path("results_clean")
CHECKPOINT_DIR = Path("models_clean")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Seed:", SEED)
print("Results:", RESULTS_DIR.resolve())
print("Checkpoints:", CHECKPOINT_DIR.resolve())

## 2. Dataset pipeline and fixed split

We use the official CIFAR-10 training partition (50,000 images) to construct one fixed 45,000-image training pool and 5,000-image validation set. The official 10,000-image test set remains untouched.

This split is defined once with a seeded PyTorch generator. Training subsets at 10%, 25%, 50%, and 100% are nested and drawn only from the 45,000-image training pool.

**Preprocessing note:** This baseline uses CIFAR-10 channel normalization. If the original notebook's transforms differ, do not compare its results directly with this clean pipeline without documenting the difference.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Keep transforms deterministic for a clean first sanity baseline.
# Add training augmentation only after the baseline pipeline is verified.
train_full = datasets.CIFAR10(
    root=str(DATA_ROOT), train=True, download=True, transform=eval_transform
)
test_dataset = datasets.CIFAR10(
    root=str(DATA_ROOT), train=False, download=True, transform=eval_transform
)

split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_full), generator=split_generator).tolist()
train_indices = permutation[:45000]
validation_indices = permutation[45000:]

train_pool = Subset(train_full, train_indices)
validation_dataset = Subset(train_full, validation_indices)

subset_order = torch.randperm(
    len(train_pool), generator=torch.Generator().manual_seed(SEED + 1)
).tolist()

fractions = {"10%": 0.10, "25%": 0.25, "50%": 0.50, "100%": 1.00}
train_subsets = {
    name: Subset(train_pool, subset_order[:int(len(train_pool) * fraction)])
    for name, fraction in fractions.items()
}

print("Train pool:", len(train_pool))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))
print("Subset sizes:", {k: len(v) for k, v in train_subsets.items()})
assert len(train_pool) == 45000
assert len(validation_dataset) == 5000
assert len(test_dataset) == 10000
assert set(train_indices).isdisjoint(set(validation_indices))
assert len(set(train_indices) | set(validation_indices)) == len(train_full)

## 3. Data integrity checks

In [ ]:
def get_targets(dataset):
    # Handles Subset objects nested over CIFAR-10.
    if isinstance(dataset, Subset):
        parent_targets = get_targets(dataset.dataset)
        return [parent_targets[i] for i in dataset.indices]
    if hasattr(dataset, "targets"):
        return list(dataset.targets)
    raise TypeError(f"Cannot extract labels from {type(dataset)}")

train_targets = get_targets(train_pool)
val_targets = get_targets(validation_dataset)
test_targets = get_targets(test_dataset)

print("Train class counts:", dict(sorted(pd.Series(train_targets).value_counts().items())))
print("Validation class counts:", dict(sorted(pd.Series(val_targets).value_counts().items())))
print("Test class counts:", dict(sorted(pd.Series(test_targets).value_counts().items())))

sample_image, sample_label = train_pool[0]
print("Sample tensor shape:", tuple(sample_image.shape))
print("Sample label:", sample_label)
print("Tensor finite:", bool(torch.isfinite(sample_image).all()))
assert sample_image.shape == (3, 32, 32)
assert 0 <= int(sample_label) < 10
assert torch.isfinite(sample_image).all()
assert all(len(train_subsets[a]) <= len(train_subsets[b]) for a, b in [
    ("10%", "25%"), ("25%", "50%"), ("50%", "100%")
])
print("Data integrity checks passed.")

## 4. Data loaders

Use fixed ordering for validation and test. Training shuffle is seeded per experiment so different models can be given the same batch order when required.


In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 0

def make_loaders(train_dataset, batch_size=BATCH_SIZE, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        generator=generator, num_workers=NUM_WORKERS
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS
    )
    return train_loader, validation_loader

validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

xb, yb = next(iter(validation_loader))
print("Validation batch:", tuple(xb.shape), tuple(yb.shape))

## 5. Canonical classical ViT

In [ ]:
class ClassicalViT(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=4,
        in_channels=3,
        embed_dim=128,
        depth=4,
        num_heads=4,
        mlp_dim=256,
        num_classes=10,
        dropout=0.1,
    ):
        super().__init__()
        assert image_size % patch_size == 0
        num_patches = (image_size // patch_size) ** 2

        self.patch_embedding = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.embedding_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=depth
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

    def forward_features(self, images):
        tokens = self.patch_embedding(images)
        tokens = tokens.flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(images.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = tokens + self.pos_embedding
        tokens = self.embedding_dropout(tokens)
        tokens = self.transformer(tokens)
        return self.norm(tokens[:, 0])

    def forward(self, images):
        return self.classifier(self.forward_features(images))

set_seed()
classical_vit = ClassicalViT().to(DEVICE)
n_params = sum(p.numel() for p in classical_vit.parameters())
print("ClassicalViT parameters:", f"{n_params:,}")
with torch.no_grad():
    test_logits = classical_vit(xb[:4].to(DEVICE))
print("Smoke-test logits shape:", tuple(test_logits.shape))
assert test_logits.shape == (4, 10)
assert torch.isfinite(test_logits).all()
print("Forward smoke test passed.")

## 6. Shared train/evaluation utilities

This training loop reports sample-weighted loss and accuracy. It never uses validation labels for gradient updates. Checkpoints are written to `models_clean/`, separate from existing research checkpoints.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).long()

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        batch_n = labels.size(0)
        total_loss += loss.item() * batch_n
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_n

    return {
        "loss": total_loss / max(total_examples, 1),
        "accuracy": total_correct / max(total_examples, 1),
        "n": total_examples,
    }

def fit_model(model, train_loader, validation_loader, epochs=3,
              learning_rate=1e-3, weight_decay=1e-4, run_name="sanity"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    history = []

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, validation_loader, criterion, optimizer=None)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "validation_loss": val_metrics["loss"],
            "validation_accuracy": val_metrics["accuracy"],
            "seconds": time.time() - start,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {row['train_loss']:.4f} acc {row['train_accuracy']:.3f} | "
            f"val loss {row['validation_loss']:.4f} acc {row['validation_accuracy']:.3f} | "
            f"{row['seconds']:.1f}s"
        )

    history_df = pd.DataFrame(history)
    history_df.to_csv(RESULTS_DIR / f"{run_name}_history.csv", index=False)
    torch.save(
        {"model_state_dict": model.state_dict(), "history": history,
         "run_name": run_name, "seed": SEED},
        CHECKPOINT_DIR / f"{run_name}_last.pt"
    )
    return history_df

## 7. Classical ViT sanity test — small fixed subset

Run this before HQViT. It is a **sanity test**, not a final benchmark. We use a fixed 256-image subset and a short run to check whether the canonical model can learn at all. If training accuracy does not clearly improve, stop and investigate before adding quantum components.

This is intentionally not a full 45,000-image experiment.


In [ ]:
SANITY_N = 256
SANITY_EPOCHS = 5
SANITY_BATCH_SIZE = 32
sanity_indices = torch.randperm(
    len(train_pool), generator=torch.Generator().manual_seed(SEED + 2)
)[:SANITY_N].tolist()
sanity_subset = Subset(train_pool, sanity_indices)

sanity_train_loader, sanity_val_loader = make_loaders(
    sanity_subset, batch_size=SANITY_BATCH_SIZE, seed=SEED + 3
)

set_seed(SEED)
sanity_model = ClassicalViT().to(DEVICE)
sanity_history = fit_model(
    sanity_model,
    sanity_train_loader,
    sanity_val_loader,
    epochs=SANITY_EPOCHS,
    learning_rate=1e-3,
    weight_decay=1e-4,
    run_name="classical_vit_256_sanity",
)

print("\nSanity-test summary")
print("Initial train accuracy:", round(float(sanity_history.iloc[0]["train_accuracy"]), 4))
print("Final train accuracy:", round(float(sanity_history.iloc[-1]["train_accuracy"]), 4))
print("Initial validation accuracy:", round(float(sanity_history.iloc[0]["validation_accuracy"]), 4))
print("Final validation accuracy:", round(float(sanity_history.iloc[-1]["validation_accuracy"]), 4))
print("Interpret cautiously: five epochs may be insufficient; this is a pipeline check.")

## 8. Gate before proceeding

Do not begin HQViT comparisons until the sanity-test behavior has been reviewed.

A small subset should generally show clear training improvement, but there is no universal accuracy threshold for this short run. If the loss does not decrease or accuracy remains near chance, inspect the data pipeline, gradients, optimizer updates, and model initialization first.

## 9. HQViT implementation — intentionally not included yet

Only after the classical sanity test is understood should the verified quantum circuit and hybrid model be migrated here. This prevents a second copy of the old notebook's competing definitions from silently becoming the canonical implementation.

## Experiment log template

For each run, record: code version/commit, split seed, subset indices or deterministic seed, preprocessing, architecture, parameter count, optimizer, learning rate, weight decay, batch size, epochs, random seed, best validation metric, checkpoint path, and any deviations.
